# 99 Duplicates: are the rows we drop really duplicates?
The last step of `02_data_cleaning.ipynb` drops duplicates: two rows count as the same listing when **every column except `state`** matches (price, year, model, odometer, condition, colour, ...). `state` is left out because the same car is often posted in several regions / states.

That's an assumption, not a proof. This notebook checks it with what the dedupe **never looked at**:
- **VIN**: same VIN = same car, different VIN = two different cars
- **description**: when a VIN is missing, the same ad text means the same listing

It also checks the other direction: reposts the dedupe **misses** (same car, but the price or odometer changed).

Then: **why** the duplicates exist, three example cars posted in 4 regions, and whether the ad text and the model text are really the same in every post.

## 1 Replay the cleaning up to the dedupe
Runs the code cells of `02_data_cleaning.ipynb` up to (not including) the `drop_duplicates` cell, so `df` is exactly what the dedupe sees. Nothing is saved.

In [16]:
import json
import numpy as np
import pandas as pd

for cell in json.load(open("02_data_cleaning.ipynb"))["cells"]:
    src = "".join(cell["source"])
    if cell["cell_type"] != "code":
        continue
    if "drop_duplicates" in src:
        break
    exec(src)
df.shape

(357696, 19)

## 2 Pair each removed row with the row it matched
- same rule as `02_data_cleaning.ipynb`: every column except `state` and the raw listing fields (url, VIN, description, lat / long)
- the dedupe keeps the first row of each group, so every removed row matches exactly one kept row

In [17]:
dedupe_cols = [c for c in df.columns if c not in ["state"] + RAW_LISTING_COLS]
removed = df[df.duplicated(subset=dedupe_cols)]
kept = df.drop_duplicates(subset=dedupe_cols)
pairs = removed.merge(kept, on=dedupe_cols, suffixes=("_removed", "_kept"))
assert len(pairs) == len(removed)

print(f"{len(df):,} rows before the dedupe, {len(removed):,} removed ({len(removed) / len(df):.1%})")

357,696 rows before the dedupe, 152,311 removed (42.6%)


## 3 Evidence: was each removed row really the same car?
- VINs are only compared when both rows have a real one (placeholders like `11111111111111111` don't identify a car)
- no VIN on one side → compare the description text

In [18]:
def real_vin(vin):
    return (vin.str.len() == 17) & ~vin.str.fullmatch(r"(.)\1+", na=False)

vin_removed, vin_kept = pairs["VIN_removed"].str.upper(), pairs["VIN_kept"].str.upper()
both_vins = real_vin(vin_removed) & real_vin(vin_kept)
same_vin = both_vins & (vin_removed == vin_kept)
same_text = pairs["description_removed"] == pairs["description_kept"]

pairs["evidence"] = np.select(
    [same_vin, both_vins, same_text],
    ["same VIN: same car", "different VIN: different cars", "no VIN to compare, same description"],
    default="no VIN to compare, different description",
)
evidence = pairs["evidence"].value_counts().to_frame("pairs")
evidence["share"] = (evidence["pairs"] / evidence["pairs"].sum()).round(3)
evidence

,pairs,share
evidence,,
same VIN: same car,120744,0.793
"no VIN to compare, same description",19790,0.130
"no VIN to compare, different description",11717,0.077
different VIN: different cars,60,0.000


**How to read it:**
- **same VIN** and **same description** → the dedupe removed a real duplicate
- **different VIN** → two different cars that match on every column, i.e. wrongly removed. Usually a dealer with several identical cars at the same price (checked below). For the model these rows are identical anyway (same features, same price), so losing them costs little
- **no VIN, different description** → can't tell from this data; eyeball the examples below

## 4 Examples
Different VIN (wrongly removed?) and whether they're dealer listings (craigslist `ctd` = dealer):

In [19]:
wrong = pairs[pairs["evidence"] == "different VIN: different cars"]
print(f"dealer listings among them: {wrong['url_removed'].str.contains('/ctd/').mean():.0%}")
wrong[["price", "year", "manufacturer", "model", "odometer", "state_removed", "state_kept", "VIN_removed", "VIN_kept"]].head(10)

dealer listings among them: 97%


,price,year,manufacturer,model,odometer,state_removed,state_kept,VIN_removed,VIN_kept
5652,20999,2013,ford,f-150,102909.0,ca,ca,1FTFX1ET7DFC00259,1FTFX1ET7DFC00258
5654,10999,2006,chevrolet,silverado 1500,112512.0,ca,ca,1GCEK14X16Z136879,1GCEK14X16Z136826
15746,3995,2005,hyundai,tucson,160083.0,ca,ca,KM8JN12D85U042715,KM8JN12D85U042751
15856,3995,2005,hyundai,tucson,160083.0,ca,ca,KM8JN12D85U042715,KM8JN12D85U042751
18308,3995,2005,hyundai,tucson,160083.0,ca,ca,KM8JN12D85U042715,KM8JN12D85U042751
18378,20999,2013,ford,f-150,102909.0,ca,ca,1FTFX1ET7DFC00259,1FTFX1ET7DFC00258
18382,10999,2006,chevrolet,silverado 1500,112512.0,ca,ca,1GCEK14X16Z136879,1GCEK14X16Z136826
18418,3995,2005,hyundai,tucson,160083.0,ca,ca,KM8JN12D85U042715,KM8JN12D85U042751
18544,3995,2005,hyundai,tucson,160083.0,ca,ca,KM8JN12D85U042715,KM8JN12D85U042751
18633,3995,2005,hyundai,tucson,160083.0,ca,ca,KM8JN12D85U042715,KM8JN12D85U042751


No VIN and a different description (the unsure ones), with the start of both ads:

In [20]:
unsure = pairs[pairs["evidence"] == "no VIN to compare, different description"]
unsure.assign(
    ad_removed=unsure["description_removed"].str[:80],
    ad_kept=unsure["description_kept"].str[:80],
)[["price", "year", "manufacturer", "model", "odometer", "ad_removed", "ad_kept"]].head(10)

,price,year,manufacturer,model,odometer,ad_removed,ad_kept
12,6800,2005,unknown,unknown,180000.0,"2005 Blue Bird School Bus: 12 window, 72 passe...","2005 Blue Bird 12 window, 72 passenger bus tha..."
38,16888,2008,toyota,fj cruiser,188096.0,JUST TRADED CLEAN CAR FAX. VERY NICE .. PLEASE...,JUST TRADED CLEAN CAR FAX. VERY NICE .. GARYEJ...
551,21250,2002,ford,thunderbird,15962.0,2002 FORD THUNDERBIRD PREMIUM DELUXE EDITION I...,2002 ford thunderbird premium deluxe edition i...
590,4500,2014,ford,f-250,250000.0,NOTE: Truck Is Located In Eva AL. 30 mins sout...,2014 Ford F-250 XL 2wd 6.2L Mechanic Special ...
607,12250,2007,toyota,4runner,189000.0,No Trades!!! This 4Runner Is Located In Cullma...,Please Read The Full Ad Prior To Contacting Me...
609,5500,2007,acura,mdx,158000.0,"2007 Acura Runs great, 158k miles, Rebuilt ti...","2007 Acura Runs great, 158k miles, Rebuilt ti..."
625,10400,2006,dodge,other,89200.0,"ONLY 89,000 MILES!!! QUAD/CREW 4 DOOR LOADED! ...","ONLY 89,000 MILES!!! QUAD 4 DOOR LOADED! 2 Own..."
634,6900,2014,ford,fusion,116000.0,"2014 Ford Fusion SE Runs good, Only 116k miles...","2014 Ford Fusion SE Runs good, Only 116k miles..."
676,20288,2015,mercedes-benz,glk-class,48889.0,2015 Mercedes Benz GLK 350 Blind Spot Assist P...,2015 Mercedes Benz GLK 350 Blind Spot Assist P...
710,6950,2011,nissan,quest,54443.0,DEALER INFORMATION Dealer Name:\tBell Auto Sal...,DEALER INFORMATION Dealer Name:\tBell Auto Sal...


## 5 The other direction: reposts the dedupe misses
A car reposted with a new price or odometer is not an exact match, so it stays in the data. Rows that still share a real VIN after the dedupe are the same car listed more than once. They matter because copies can land in both train and test, which makes the test score a bit optimistic.

In [21]:
vin = kept["VIN"].str.upper()
repost = real_vin(vin) & vin.duplicated(keep=False)
print(f"after the dedupe, {repost.sum():,} rows still share a VIN with another row ({vin[repost].nunique():,} cars)")

# for those cars: how often each column differs between the copies
groups = kept[repost].assign(VIN=vin[repost]).groupby("VIN")
(groups[["price", "odometer", "state", "condition"]].nunique() > 1).mean().round(2).to_frame("share of cars where it differs")

after the dedupe, 8,170 rows still share a VIN with another row (3,887 cars)


,share of cars where it differs
price,0.55
odometer,0.12
state,0.24
condition,0.13


In [22]:
kept[repost].assign(VIN=vin[repost]).sort_values("VIN")[["VIN", "price", "odometer", "year", "manufacturer", "model", "state"]].head(12)

,VIN,price,odometer,year,manufacturer,model,state
10916,137ZA903XYE188501,119999,56322.0,2000,unknown,other,az
45069,137ZA903XYE188501,74877,56322.0,2000,unknown,other,ca
15164,19UUA66217A023268,4900,171390.0,2007,acura,tl,az
15790,19UUA66217A023268,5900,171390.0,2007,acura,tl,az
42469,19UUA66248A022729,7799,117567.0,2008,acura,other,ca
56686,19UUA66248A022729,7799,117000.0,2008,acura,tl,ca
297145,19UUA66267A015313,6850,163080.0,2007,acura,tl,oh
295650,19UUA66267A015313,5950,163080.0,2007,acura,tl,oh
335452,19UUA66276A061151,5695,125036.0,2006,acura,tl,pa
338169,19UUA66276A061151,5995,125036.0,2006,acura,tl,pa


## 6 Why: who posts the same car more than once
Craigslist is one website split into ~400 **regional sites** (`sarasota.craigslist.org`, `columbus.craigslist.org`, ...). A post lives in one region, and buyers mostly browse their own, so a seller who wants buyers in more cities has to post the car again in each region. The url of every row has the region and the post id.

Each removed row is compared with the row it matched:
- **same post id** → the scraper picked up the same post twice (a scraping bug)
- **posted in another region** (same state or another state) → cross-posting to reach more buyers
- **reposted in the same region** → posting again later to get back to the top of the list (newest first)

In [23]:
raw = pd.read_csv("data/vehicles.csv", usecols=["url", "model", "posting_date"]).drop_duplicates("url").set_index("url")

for s in ["_removed", "_kept"]:
    pairs["post" + s] = pairs["url" + s].str.extract(r"/(\d+)\.html")[0]
    pairs["region" + s] = pairs["url" + s].str.extract(r"https?://([^.]+)\.craigslist")[0]
    pairs["date" + s] = pd.to_datetime(pairs["url" + s].map(raw["posting_date"]), utc=True)

same_post = pairs["post_removed"] == pairs["post_kept"]
pairs["why"] = np.select(
    [same_post, pairs["region_removed"] == pairs["region_kept"], pairs["state_removed"] == pairs["state_kept"]],
    ["same post scraped twice", "reposted in the same region", "posted in another region, same state"],
    default="posted in another state",
)
pairs["dealer"] = pairs["url_removed"].str.contains("/ctd/")
pairs["carvana"] = pairs["description_removed"].str.contains("Carvana", na=False)
days_apart = (pairs["date_removed"] - pairs["date_kept"]).abs().dt.total_seconds() / 86400

print(f"same post scraped twice: {same_post.sum():,} rows")
print(f"dealer listings: {pairs['dealer'].mean():.0%} of removed rows | Carvana alone: {pairs['carvana'].mean():.0%}")
by_why = pairs.groupby("why")
pd.DataFrame({
    "rows": by_why.size(),
    "share": (by_why.size() / len(pairs)).round(3),
    "dealer": by_why["dealer"].mean().round(2),
    "carvana": by_why["carvana"].mean().round(2),
    "median days apart": days_apart.groupby(pairs["why"]).median().round(1),
}).sort_values("rows", ascending=False)

same post scraped twice: 0 rows
dealer listings: 97% of removed rows | Carvana alone: 35%


,rows,share,dealer,carvana,median days apart
why,,,,,
posted in another state,86819,0.570,0.99,0.58,0.2
"posted in another region, same state",34664,0.228,0.97,0.10,0.2
reposted in the same region,30828,0.202,0.92,0.00,7.1


**How to read it:** copies in another region a few hours apart = cross-posting (Carvana ships nationwide, so it posts its whole inventory everywhere); copies in the same region about a week apart = bumping. Private owners almost never do either, so nearly every duplicate comes from a dealer.

## 7 Examples: cars posted in 4 regions
Cars with a real VIN, one of each kind: Carvana, a local dealer, a private owner. For every post: when (local time), where, price, mileage, the post id, the model **exactly as the seller typed it**, and whether the ad text is identical to the first post.

In [24]:
posts = df.assign(
    VIN=df["VIN"].str.upper(),
    region=df["url"].str.extract(r"https?://([^.]+)\.craigslist")[0],
    post=df["url"].str.extract(r"/(\d+)\.html")[0],
    posted=df["url"].map(raw["posting_date"]),
    model_raw=df["url"].map(raw["model"]),
    seller=np.where(df["url"].str.contains("/ctd/"), "dealer", "owner"),
    carvana=df["description"].str.contains("Carvana", na=False),
)
posts = posts[real_vin(posts["VIN"])]
posts["date"] = pd.to_datetime(posts["posted"], utc=True)

def show_car(vin):
    car = posts[posts["VIN"] == vin].sort_values("date")
    print(f"{car['year'].iloc[0]} {car['manufacturer'].iloc[0]} {car['model'].iloc[0]} | VIN {vin} | "
          f"{len(car)} posts in {car['region'].nunique()} regions, {car['state'].nunique()} states")
    print("first url:", car["url"].iloc[0])
    return car.assign(posted=car["posted"].str[:16], same_ad_as_first=car["description"] == car["description"].iloc[0])[
        ["posted", "region", "state", "seller", "price", "odometer", "post", "model_raw", "same_ad_as_first"]]

**Carvana, 2020 Honda Passport.** The url says `atlanta-2020-honda-passport`: the car is in Atlanta, but it's advertised in California, Michigan and Maryland.

In [25]:
show_car("5FNYF8H01LB009820")

2020 honda passport | VIN 5FNYF8H01LB009820 | 4 posts in 4 regions, 3 states
first url: https://ventura.craigslist.org/ctd/d/atlanta-2020-honda-passport-elite-sport/7301963717.html


,posted,region,state,seller,price,odometer,post,model_raw,same_ad_as_first
69051,2021-04-05T04:21,ventura,ca,dealer,40990,5833.0,7301963717,passport elite sport,True
59700,2021-04-05T05:14,santabarbara,ca,dealer,40990,5833.0,7301975564,passport elite sport,False
208392,2021-04-18T09:33,porthuron,mi,dealer,40990,5833.0,7308694543,passport elite sport,False
182818,2021-04-18T12:00,baltimore,md,dealer,40990,5833.0,7308767692,passport elite sport,False


**Local dealer, 2021 Chevrolet Trailblazer.** A dealer near the PA / OH border reposting about once a week, each time in a nearby city's region.

In [26]:
show_car("KL79MMS2XMB026016")

2021 chevrolet trailblazer | VIN KL79MMS2XMB026016 | 4 posts in 4 regions, 2 states
first url: https://harrisburg.craigslist.org/ctd/d/darlington-2021-chevrolet-trailblazer/7303624197.html


,posted,region,state,seller,price,odometer,post,model_raw,same_ad_as_first
329873,2021-04-08T10:43,harrisburg,pa,dealer,12500,2773.0,7303624197,trailblazer,True
293981,2021-04-16T10:35,cleveland,oh,dealer,12500,2773.0,7307688386,trailblazer,False
336255,2021-04-29T11:50,pittsburgh,pa,dealer,12500,2773.0,7314340999,trailblazer,False
303015,2021-05-04T10:36,youngstown,oh,dealer,12500,2773.0,7316705412,trailblazer,False


**Private owner, 2020 Chevrolet Silverado 3500.** Four posts in 20 minutes, but the first one has a different price, so it isn't an exact match: the dedupe keeps it, and this truck is still in the data twice (one of the reposts from section 5).

In [27]:
show_car("1GB3WREY1LF301140")

2020 chevrolet silverado 3500 | VIN 1GB3WREY1LF301140 | 4 posts in 4 regions, 2 states
first url: https://charlotte.craigslist.org/cto/d/greensboro-2020-chevrolet-silverado/7316414731.html


,posted,region,state,seller,price,odometer,post,model_raw,same_ad_as_first
231838,2021-05-03T16:38,charlotte,nc,owner,59500,1025.0,7316414731,silverado 3500hd,True
236676,2021-05-03T16:44,greensboro,nc,owner,59900,1025.0,7316418474,silverado 3500hd,True
240459,2021-05-03T16:50,raleigh,nc,owner,59900,1025.0,7316422032,silverado 3500hd,True
395828,2021-05-03T16:58,richmond,va,owner,59900,1025.0,7316426460,silverado 3500hd,True


**The most posted car in the data:**

In [28]:
most = show_car(posts.groupby("VIN")["region"].nunique().idxmax())
print("posted", most["posted"].min(), "to", most["posted"].max(), "| prices:", most["price"].unique())
most.head()

2017 ford expedition | VIN 1FMJU1JT1HEA52352 | 261 posts in 261 regions, 38 states
first url: https://lascruces.craigslist.org/ctd/d/las-cruces-2017-ford-expedition-xlt/7311766797.html
posted 2021-04-24T03:35 to 2021-04-24T15:01 | prices: [29590]


,posted,region,state,seller,price,odometer,post,model_raw,same_ad_as_first
262390,2021-04-24T04:35,lascruces,nm,dealer,29590,70760.0,7311766797,expedition xlt sport,True
60039,2021-04-24T03:35,santamaria,ca,dealer,29590,70760.0,7311766813,expedition xlt sport,False
37938,2021-04-24T03:35,modesto,ca,dealer,29590,70760.0,7311766823,expedition xlt sport,False
65049,2021-04-24T03:35,stockton,ca,dealer,29590,70760.0,7311766828,expedition xlt sport,False
190440,2021-04-24T06:35,westernmass,ma,dealer,29590,70760.0,7311766848,expedition xlt sport,False


## 8 Is the text really the same in every post?
For every car with a real VIN posted 2+ times:
- **model text**: the model exactly as the seller typed it (before the model-name cleaning in `01_data_cleaning_model_names.ipynb`)
- **ad text**: the full description

Exact comparison is strict: one changed character (a local phone number, a tracking code) makes two ads "different". So the ad text also gets a **similarity score** (difflib, 1.0 = identical), on a sample of cars because it's slow on long ads.

In [29]:
multi = posts[posts.groupby("VIN")["VIN"].transform("size") > 1]
cars = multi.groupby("VIN")
same = pd.DataFrame({
    "kind": np.where(cars["carvana"].any(), "Carvana", np.where(cars["seller"].first() == "dealer", "other dealer", "private owner")),
    "same model text": cars["model_raw"].nunique(dropna=False) == 1,
    "same ad text": cars["description"].nunique(dropna=False) == 1,
}, index=cars.size().index)

summary = same.groupby("kind")[["same model text", "same ad text"]].mean().round(2)
summary["cars"] = same.groupby("kind").size()
summary

,same model text,same ad text,cars
kind,,,
Carvana,1.00,0.00,5738
other dealer,0.94,0.30,25757
private owner,0.83,0.52,483


In [30]:
import difflib

# first post vs up to 5 later posts, for 300 random cars
scores = []
for vin, car in multi[multi["VIN"].isin(same.sample(300, random_state=0).index)].groupby("VIN"):
    first, *later = car.sort_values("date")["description"].fillna("").tolist()
    for other in later[:5]:
        scores.append({"kind": same.loc[vin, "kind"], "similarity": difflib.SequenceMatcher(None, first, other).ratio()})
pd.DataFrame(scores).groupby("kind")["similarity"].describe()[["count", "min", "50%", "mean"]].round(3)

,count,min,50%,mean
kind,,,,
Carvana,147.0,0.872,0.994,0.992
other dealer,515.0,0.025,0.995,0.944
private owner,5.0,0.891,0.966,0.961


**What actually changes** between two posts of the Carvana Passport (each difference with some text around it):

In [31]:
ads = posts[posts["VIN"] == "5FNYF8H01LB009820"].sort_values("date")["description"].tolist()
sm = difflib.SequenceMatcher(None, ads[0], ads[2])
pd.DataFrame([
    {"first post": ads[0][max(a1 - 25, 0):a2 + 25], "later post": ads[2][max(b1 - 25, 0):b2 + 25]}
    for op, a1, a2, b1, b2 in sm.get_opcodes() if op != "equal"
])

,first post,later post
0,ailability? Call us at 805-491-1125Just text ...,ailability? Call us at 810-310-8617Just text ...
1,ability? Call us at 805-491-1125Just text tha...,ability? Call us at 810-310-8617Just text tha...
2,ity? Call us at 805-491-1125Just text that st...,ity? Call us at 810-310-8617Just text that st...
3,carvanaauto.com/6834572-75215 and plug it into...,carvanaauto.com/6834572-74671 and plug it into...
4,carvanaauto.com/6834572-75215Looking for more ...,carvanaauto.com/6834572-74671Looking for more ...
5,like this one? We have 16 Honda Passport in s...,like this one? We have 14 Honda Passport in s...
6,n stock for as low as $27590!Why buy with Carv...,n stock for as low as $27990!Why buy with Carv...
7,carvanaauto.com/6834572-75215NO REPORTED ACCID...,carvanaauto.com/6834572-74671NO REPORTED ACCID...
8,carvanaauto.com/6834572-75215COST SAVINGS: Car...,carvanaauto.com/6834572-74671COST SAVINGS: Car...
9,uestion with Carvana at 805-491-1125*Including...,uestion with Carvana at 810-310-8617*Including...


In [32]:
import importlib, carpricepredictor.shared
importlib.reload(carpricepredictor.shared)

<module 'carpricepredictor.shared' from '/Users/hugo/Documents/GitHub/CarPricePredictor/src/carpricepredictor/shared.py'>

**How to read it:** the model text is copy-pasted, so it's the same in almost every post. The ad text is rarely identical to the character, but a similarity close to 1.0 means the ad is copy-pasted too and only small bits change per region (phone number, tracking code, link). Either way, it's the same car and the same ad.

**Possible fix (not applied yet):** in `02_data_cleaning.ipynb`, also keep one row per real VIN. It changes the rows, so the train / test split and every score after it change too: compare iterations only on the same data.